# NLU HOSTAGE — ai_1_nlu_v2_100

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [1]:
from pathlib import Path
DATASET_FILENAME = "v2_chat_dataset_100.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v2_100"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


Dataset aktif: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v2_100\data\v2_chat_dataset_100.csv


In [2]:
# MODUL BERSAMA: EDA + SVM + SVM TUNING + NB + NB TUNING + TRANSFORMER
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=4):
    return train_transformer(DATASET_PATH, MODEL_DIR, epochs=epochs, device=TRANSFORMER_DEVICE)

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("accusing", "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),
    ("defending", "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),
    ("bluffing", "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),
    ("probing", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),
    ("deflecting", "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),
    ("persuading", "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),
    ("claiming", "Klaim gw Civilian, gw gak punya skill malam."),
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),
train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().
Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.
Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.


In [3]:
# JALANKAN SEMUA MODEL: empat model CPU, lalu Transformer CUDA dan 10 chat uji.
RUN_TRANSFORMER = True
RUN_TUNING = False
TRANSFORMER_EPOCHS = 4
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)



MENJALANKAN: SVM baseline
SVM | train=638 | test=160 | kelas=8

--- Evaluasi SVM baseline (holdout test set) ---
Accuracy    : 0.6625
Macro F1    : 0.6545
Weighted F1 : 0.6545
              precision    recall  f1-score   support

    accusing       0.65      0.65      0.65        20
    bluffing       0.55      0.80      0.65        20
    claiming       0.50      0.30      0.38        20
   defending       0.65      0.65      0.65        20
  deflecting       0.63      0.85      0.72        20
     neutral       0.92      0.60      0.73        20
  persuading       0.61      0.55      0.58        20
     probing       0.86      0.90      0.88        20

    accuracy                           0.66       160
   macro avg       0.67      0.66      0.65       160
weighted avg       0.67      0.66      0.65       160



Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v2_100\models\intent_classifier_svm.pkl

MENJALANKAN: Naive Bayes baseline
Naive Bayes | train=638 | test=160 | kelas=8

--- Evaluasi Naive Bayes baseline (holdout test set) ---
Accuracy    : 0.6062
Macro F1    : 0.5941
Weighted F1 : 0.5941
              precision    recall  f1-score   support

    accusing       0.63      0.60      0.62        20
    bluffing       0.60      0.75      0.67        20
    claiming       0.71      0.50      0.59        20
   defending       0.55      0.60      0.57        20
  deflecting       0.48      0.70      0.57        20
     neutral       1.00      0.30      0.46        20
  persuading       0.53      0.45      0.49        20
     probing       0.68      0.95      0.79        20

    accuracy                           0.61       160
   macro avg       0.65      0.61      0.59       160
weighted avg       0.65      0.61      0.59       160

Model tersimpan: C:\Users\an

C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Map:   0%|          | 0/638 [00:00<?, ? examples/s]


Map: 100%|██████████| 638/638 [00:00<00:00, 33998.65 examples/s]

...

expected=defending    | predicted=defending    | confidence= 39.66% | OK
expected=bluffing     | predicted=bluffing     | confidence= 48.55% | OK
expected=probing      | predicted=probing      | confidence= 56.79% | OK
expected=deflecting   | predicted=deflecting   | confidence= 22.07% | OK
expected=persuading   | predicted=accusing     | confidence= 28.66% | MISS
expected=claiming     | predicted=bluffing     | confidence= 27.27% | MISS
expected=neutral      | predicted=neutral      | confidence= 33.60% | OK
expected=accusing     | predicted=deflecting   | confidence= 28.63% | MISS
expected=defending    | predicted=probing      | confidence= 22.06% | MISS

--- 10 chat uji: IndoBERT Transformer ---


expected=accusing     | predicted=persuading   | confidence= 85.23% | MISS
expected=defending    | predicted=defending    | confidence= 96.29% | OK
expected=bluffing     | predicted=defending    | confidence= 92.68% | MISS
expected=probing      | predicted=probing      | confidence= 91.16% | OK
expected=deflecting   | predicted=defending    | confidence= 84.31% | MISS
expected=persuading   | predicted=persuading   | confidence= 53.84% | OK
expected=claiming     | predicted=defending    | confidence= 94.88% | MISS
expected=neutral      | predicted=defending    | confidence= 91.77% | MISS
expected=accusing     | predicted=persuading   | confidence= 65.00% | MISS
expected=defending    | predicted=defending    | confidence= 95.59% | OK
RINGKASAN EVALUASI HOLDOUT:


,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,waktu_detik,status
0,IndoBERT Transformer,0.9437,0.9432,0.9432,32.8,berhasil
1,SVM baseline,0.6625,0.6545,0.6545,0.2,berhasil
2,Naive Bayes baseline,0.6062,0.5941,0.5941,0.1,berhasil


RINGKASAN 10 CHAT UJI:


,model,benar_dari_10,akurasi_10_chat
0,SVM baseline,7,0.7
1,Naive Bayes baseline,6,0.6
2,IndoBERT Transformer,4,0.4


## Laporan eksekusi notebook

Tuning SVM dan Naive Bayes dilewati untuk mempercepat run ini. Output training lengkap tersimpan pada cell tepat di atas.

| Model | Macro-F1 holdout | Uji 10 chat |
|---|---:|---:|
| SVM baseline | 0.6545 | 7/10 |
| Naive Bayes baseline | 0.5941 | 6/10 |
| IndoBERT Transformer (GPU) | 0.9432 | 4/10 |

SVM baseline lebih stabil untuk chat baru; Transformer sangat tinggi di holdout tetapi belum stabil di chat manual.